In [0]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully")

Libraries loaded successfully


In [0]:
DATA_PATH = "demand_forecasting.csv"

df = pd.read_csv(DATA_PATH)

print("Original rows:", len(df))
print("Original columns:", len(df.columns))

display(df.head())

Original rows: 76000
Original columns: 16


Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [0]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Data sorted successfully")

display(
    df[
        [
            "Date",
            "Store ID",
            "Product ID",
            "Demand"
        ]
    ].head(20)
)

Data sorted successfully


Date,Store ID,Product ID,Demand
2022-01-01T00:00:00.000Z,S001,P0001,115
2022-01-02T00:00:00.000Z,S001,P0001,84
2022-01-03T00:00:00.000Z,S001,P0001,132
2022-01-04T00:00:00.000Z,S001,P0001,67
2022-01-05T00:00:00.000Z,S001,P0001,110
2022-01-06T00:00:00.000Z,S001,P0001,146
2022-01-07T00:00:00.000Z,S001,P0001,87
2022-01-08T00:00:00.000Z,S001,P0001,113
2022-01-09T00:00:00.000Z,S001,P0001,87
2022-01-10T00:00:00.000Z,S001,P0001,99


In [0]:
GROUP_COLS = ["Store ID","Product ID"]

print("Time-series grouping columns:", GROUP_COLS)

Time-series grouping columns: ['Store ID', 'Product ID']


In [0]:
df["day_of_week"] = df["Date"].dt.dayofweek
df["day_of_month"] = df["Date"].dt.day
df["week_of_year"] = df["Date"].dt.isocalendar().week.astype(int)
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter
df["year"] = df["Date"].dt.year

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

display(
    df[
        [
            "Date",
            "day_of_week",
            "month",
            "quarter",
            "year",
            "is_weekend"
        ]
    ].head()
)

Date,day_of_week,month,quarter,year,is_weekend
2022-01-01T00:00:00.000Z,5,1,1,2022,1
2022-01-02T00:00:00.000Z,6,1,1,2022,1
2022-01-03T00:00:00.000Z,0,1,1,2022,0
2022-01-04T00:00:00.000Z,1,1,1,2022,0
2022-01-05T00:00:00.000Z,2,1,1,2022,0


In [0]:
GROUP_COLS = ["Store ID", "Product ID"]

df["lag_1"] = (df.groupby(GROUP_COLS)["Demand"].shift(1))

df["lag_7"] = (df.groupby(GROUP_COLS)["Demand"].shift(7))

df["lag_14"] = (df.groupby(GROUP_COLS)["Demand"].shift(14))

df["lag_28"] = (df.groupby(GROUP_COLS)["Demand"].shift(28))

display(
    df[
        [
            "Date",
            "Store ID",
            "Product ID",
            "Demand",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28"
        ]
    ].head(35)
)

Date,Store ID,Product ID,Demand,lag_1,lag_7,lag_14,lag_28
2022-01-01T00:00:00.000Z,S001,P0001,115,null,null,null,null
2022-01-02T00:00:00.000Z,S001,P0001,84,115.0,null,null,null
2022-01-03T00:00:00.000Z,S001,P0001,132,84.0,null,null,null
2022-01-04T00:00:00.000Z,S001,P0001,67,132.0,null,null,null
2022-01-05T00:00:00.000Z,S001,P0001,110,67.0,null,null,null
2022-01-06T00:00:00.000Z,S001,P0001,146,110.0,null,null,null
2022-01-07T00:00:00.000Z,S001,P0001,87,146.0,null,null,null
2022-01-08T00:00:00.000Z,S001,P0001,113,87.0,115.0,null,null
2022-01-09T00:00:00.000Z,S001,P0001,87,113.0,84.0,null,null
2022-01-10T00:00:00.000Z,S001,P0001,99,87.0,132.0,null,null


In [0]:
df["rolling_mean_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

df["rolling_mean_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(14).mean()
      )
)

df["rolling_mean_28"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(28).mean()
      )
)

In [0]:
df["rolling_std_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(7).std()
      )
)

df["rolling_std_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(14).std()
      )
)

In [0]:
feature_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

display(
    df[feature_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)


missing_values
2800
2800
1400
1400
1400
700
700
700
100


In [0]:
df_model = df.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28"
    ]
).copy()

print("Original rows:", len(df))
print("Model rows:", len(df_model))

Original rows: 76000
Model rows: 73200


In [0]:
CATEGORICAL_FEATURES = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
    "Epidemic"
]

print("Categorical features:")
print(CATEGORICAL_FEATURES)

Categorical features:
['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality', 'Epidemic']


In [0]:
BUSINESS_FEATURES = [
    "Inventory Level",
    "Units Ordered",
    "Price",
    "Discount",
    "Promotion",
    "Competitor Pricing"
]

print("Business features:")
print(BUSINESS_FEATURES)


Business features:
['Inventory Level', 'Units Ordered', 'Price', 'Discount', 'Promotion', 'Competitor Pricing']


In [0]:
TIME_FEATURES = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend"
]

print("Time features:")
print(TIME_FEATURES)

Time features:
['day_of_week', 'day_of_month', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend']


In [0]:
LAG_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

print("Lag features:")
print(LAG_FEATURES)

Lag features:
['lag_1', 'lag_7', 'lag_14', 'lag_28']


In [0]:
ROLLING_FEATURES = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

print("Rolling features:")
print(ROLLING_FEATURES)


Rolling features:
['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_14']


In [0]:
FEATURES = (
    LAG_FEATURES
    + ROLLING_FEATURES
    + BUSINESS_FEATURES
    + TIME_FEATURES
    + CATEGORICAL_FEATURES
)

TARGET = "Demand"

print("Number of features:", len(FEATURES))

print("\nFeatures:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

Number of features: 29

Features:
- lag_1
- lag_7
- lag_14
- lag_28
- rolling_mean_7
- rolling_mean_14
- rolling_mean_28
- rolling_std_7
- rolling_std_14
- Inventory Level
- Units Ordered
- Price
- Discount
- Promotion
- Competitor Pricing
- day_of_week
- day_of_month
- week_of_year
- month
- quarter
- year
- is_weekend
- Store ID
- Product ID
- Category
- Region
- Weather Condition
- Seasonality
- Epidemic

Target: Demand


In [0]:
missing_features = [
    feature
    for feature in FEATURES
    if feature not in df_model.columns
]

if len(missing_features) == 0:
    print("All features exist successfully")
else:
    print("Missing features:")
    print(missing_features)


All features exist successfully


In [0]:
model_columns = (
    ["Date"]
    + GROUP_COLS
    + FEATURES
    + [TARGET]
)

df_model = df_model[
    model_columns
].copy()

print(
    "Final model dataset shape:",
    df_model.shape
)

display(df_model.head())


Final model dataset shape: (73200, 33)


Date,Store ID,Product ID,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7,rolling_std_14,Inventory Level,Units Ordered,Price,Discount,Promotion,Competitor Pricing,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend,Store ID,Product ID,Category,Region,Weather Condition,Seasonality,Epidemic,Demand
2022-01-29T00:00:00.000Z,S001,P0001,194.0,102.0,175.0,115.0,110.57142857142857,108.35714285714286,102.64285714285714,53.47852261377815,47.658411385863204,263,0,65.0,5,0,77.11,5,29,4,1,1,2022,1,S001,P0001,Electronics,North,Sunny,Winter,0,81
2022-01-30T00:00:00.000Z,S001,P0001,81.0,126.0,126.0,84.0,107.57142857142857,101.64285714285714,101.42857142857143,54.616411278592636,44.03077095860112,537,0,71.59,0,0,69.69,6,30,4,1,1,2022,1,S001,P0001,Electronics,North,Cloudy,Winter,0,79
2022-01-31T00:00:00.000Z,S001,P0001,79.0,44.0,102.0,132.0,100.85714285714286,98.28571428571429,101.25,54.8617309589255,43.822067801207496,464,0,72.1,10,0,70.23,0,31,5,1,1,2022,0,S001,P0001,Electronics,North,Cloudy,Winter,0,113
2022-02-01T00:00:00.000Z,S001,P0001,113.0,61.0,102.0,67.0,110.71428571428571,99.07142857142857,100.57142857142857,48.80817652031363,43.99206971491792,361,0,70.53,5,0,66.56,1,1,5,2,1,2022,0,S001,P0001,Electronics,North,Snowy,Winter,0,90
2022-02-02T00:00:00.000Z,S001,P0001,90.0,160.0,123.0,110.0,114.85714285714286,98.21428571428571,101.39285714285714,44.96453629038693,44.04748935729844,259,215,60.16,20,1,54.96,2,2,5,2,1,2022,0,S001,P0001,Electronics,North,Cloudy,Winter,0,94


In [0]:
for col in CATEGORICAL_FEATURES:

    df_model[col] = (
        df_model[col]
        .astype("category")
    )

print("Categorical columns converted successfully")


Categorical columns converted successfully


In [0]:
print("====================================")
print("FEATURE ENGINEERING SUMMARY")
print("====================================")

print("Final rows      :", len(df_model))
print("Final columns   :", len(df_model.columns))
print("Number features :", len(FEATURES))
print("Target          :", TARGET)

print(
    "Remaining nulls :",
    df_model[FEATURES + [TARGET]]
    .isnull()
    .sum()
    .sum()
)

print("====================================")

FEATURE ENGINEERING SUMMARY
Final rows      : 73200
Final columns   : 33
Number features : 29
Target          : Demand
Remaining nulls : 0


In [0]:
display(
    df_model.head(20)
)


# COMMAND ----------
# 22. Feature engineering completed

print("Feature engineering completed successfully.")

Date,Store ID,Product ID,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7,rolling_std_14,Inventory Level,Units Ordered,Price,Discount,Promotion,Competitor Pricing,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend,Store ID,Product ID,Category,Region,Weather Condition,Seasonality,Epidemic,Demand
2022-01-29T00:00:00.000Z,S001,P0001,194.0,102.0,175.0,115.0,110.57142857142857,108.35714285714286,102.64285714285714,53.47852261377815,47.658411385863204,263,0,65.0,5,0,77.11,5,29,4,1,1,2022,1,S001,P0001,Electronics,North,Sunny,Winter,0,81
2022-01-30T00:00:00.000Z,S001,P0001,81.0,126.0,126.0,84.0,107.57142857142857,101.64285714285714,101.42857142857143,54.616411278592636,44.03077095860112,537,0,71.59,0,0,69.69,6,30,4,1,1,2022,1,S001,P0001,Electronics,North,Cloudy,Winter,0,79
2022-01-31T00:00:00.000Z,S001,P0001,79.0,44.0,102.0,132.0,100.85714285714286,98.28571428571429,101.25,54.8617309589255,43.822067801207496,464,0,72.1,10,0,70.23,0,31,5,1,1,2022,0,S001,P0001,Electronics,North,Cloudy,Winter,0,113
2022-02-01T00:00:00.000Z,S001,P0001,113.0,61.0,102.0,67.0,110.71428571428571,99.07142857142857,100.57142857142857,48.80817652031363,43.99206971491792,361,0,70.53,5,0,66.56,1,1,5,2,1,2022,0,S001,P0001,Electronics,North,Snowy,Winter,0,90
2022-02-02T00:00:00.000Z,S001,P0001,90.0,160.0,123.0,110.0,114.85714285714286,98.21428571428571,101.39285714285714,44.96453629038693,44.04748935729844,259,215,60.16,20,1,54.96,2,2,5,2,1,2022,0,S001,P0001,Electronics,North,Cloudy,Winter,0,94
2022-02-03T00:00:00.000Z,S001,P0001,94.0,87.0,25.0,146.0,105.42857142857143,96.14285714285714,100.82142857142857,40.631913331178254,43.47033850420687,165,0,64.73,5,0,56.97,3,3,5,2,1,2022,0,S001,P0001,Electronics,North,Cloudy,Winter,0,66
2022-02-04T00:00:00.000Z,S001,P0001,66.0,194.0,90.0,87.0,102.42857142857143,99.07142857142857,97.96428571428571,42.92962125330692,39.50942299622339,339,0,69.7,10,1,81.05,4,4,5,2,1,2022,0,S001,P0001,Electronics,North,Snowy,Winter,0,119
2022-02-05T00:00:00.000Z,S001,P0001,119.0,81.0,102.0,113.0,91.71428571428571,101.14285714285714,99.10714285714286,18.90074324565295,39.75667750948773,263,0,64.98,5,0,76.39,5,5,5,2,1,2022,1,S001,P0001,Electronics,North,Sunny,Winter,0,81
2022-02-06T00:00:00.000Z,S001,P0001,81.0,79.0,126.0,87.0,91.71428571428571,99.64285714285714,97.96428571428571,18.90074324565295,40.116382885601446,182,124,66.85,0,0,65.43,6,6,5,2,1,2022,1,S001,P0001,Electronics,North,Sunny,Winter,0,61
2022-02-07T00:00:00.000Z,S001,P0001,61.0,113.0,44.0,99.0,89.14285714285714,95.0,97.03571428571429,21.904554864445988,40.58988127031739,117,0,61.3,5,0,70.26,0,7,6,2,1,2022,0,S001,P0001,Electronics,North,Snowy,Winter,0,65


Feature engineering completed successfully.
